In [3]:
import numpy as np
import pandas as pd

In [4]:
movies = pd.read_csv(r"tmdb_5000_movies.csv")
credits = pd.read_csv(r"tmdb_5000_credits.csv")

movies.shape

(4803, 20)

In [5]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [6]:
movies = movies.merge(credits, on='title')
movies.shape

(4809, 23)

In [ ]:
movies.head(1)

In [ ]:
movies.info()

In [7]:
movies = movies[['movie_id', 'genres', 'keywords', 'overview', 'title', 'cast', 'crew']]

In [ ]:
movies.head(1)

In [8]:
movies.isnull().sum()

movie_id    0
genres      0
keywords    0
overview    3
title       0
cast        0
crew        0
dtype: int64

In [9]:
movies.dropna(inplace=True)

In [10]:
movies.duplicated().sum()

0

In [11]:
movies['genres'].loc[0]
movies['keywords'].loc[0]

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [12]:
import ast

def convert(data):
    li = []
    for i in ast.literal_eval(data):
        li.append(i['name'])
    return li

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['cast'].loc[0]

In [13]:
def convert_cast(data):
    actors_list = []
    count = 0
    
    for i in ast.literal_eval(data):
        if count == 3:
            break
        else:
            actors_list.append(i['character'])
            count = count+1
    return actors_list

movies['cast'] = movies['cast'].apply(convert_cast)

In [ ]:
movies['crew'].loc[0]

In [14]:
def convert_crew(data):
    return [i['name'] for i in ast.literal_eval(data) if i['job'] == 'Director']

movies['crew'] = movies['crew'].apply(convert_crew)

In [15]:
movies['overview'] = movies['overview'].apply(lambda x: x.split(' '))

In [16]:
def space_remover(string_list):
    return [s.replace(" ", "") for s in string_list]


movies['genres'] = movies['genres'].apply(space_remover)
movies['keywords'] = movies['keywords'].apply(space_remover)
movies['cast'] = movies['cast'].apply(space_remover)
movies['crew'] = movies['crew'].apply(space_remover)

In [ ]:
movies.columns

In [ ]:
movies.head(2)

In [17]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head(2)

In [18]:
new_df = movies[['movie_id', 'title', 'tags']]
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [19]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [20]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [59]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [ ]:
pip install nltk

In [21]:
import nltk

In [22]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [23]:
def stem(text):
    return " ".join([ps.stem(i) for i in text.split()])

In [24]:
new_df['tags'] = new_df['tags'].apply(stem)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

In [26]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [27]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zombies', 'zone', 'zoo'], dtype=object)

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
similarity = cosine_similarity(vectors)
print(similarity)

[[1.         0.08858079 0.08718573 ... 0.04559608 0.         0.        ]
 [0.08858079 1.         0.06350006 ... 0.02490677 0.         0.0277137 ]
 [0.08718573 0.06350006 1.         ... 0.02451452 0.         0.        ]
 ...
 [0.04559608 0.02490677 0.02451452 ... 1.         0.03774257 0.04279605]
 [0.         0.         0.         ... 0.03774257 1.         0.08399211]
 [0.         0.0277137  0.         ... 0.04279605 0.08399211 1.        ]]


In [43]:
def recommend(movie):
    index = new_df[new_df['title'] == movie].index[0]
    enumerated_list = list(enumerate(similarity[index]))
    print(enumerated_list)
    distances = sorted(enumerated_list, reverse=True, key= lambda x: x[1])
    movie_list = []
    
    for i in distances[1:6]:
        movie_list.append(new_df['title'].loc[i[0]])

    print(movie_list)
        
recommend("Avatar")

[(0, 1.0), (1, 0.0885807893036239), (2, 0.08718572905786445), (3, 0.07009996372327816), (4, 0.19184045508446734), (5, 0.10195592378577323), (6, 0.037470006725398436), (7, 0.15042366316917008), (8, 0.059053859535749265), (9, 0.09208185110322616), (10, 0.10269923319261937), (11, 0.09592022754223367), (12, 0.0949157995752499), (13, 0.03958262466638396), (14, 0.1316245316231641), (15, 0.06367145399670132), (16, 0.08000711205939975), (17, 0.14273829330008253), (18, 0.09824718648648241), (19, 0.08219949365267865), (20, 0.08521145659838941), (21, 0.11277677488692017), (22, 0.06003002251876642), (23, 0.09320546490018), (24, 0.056388387443460086), (25, 0.052278773405661366), (26, 0.15404884978892902), (27, 0.18944220058835154), (28, 0.11810771907149853), (29, 0.06711560552140243), (30, 0.06590621627456203), (31, 0.14309095175803563), (32, 0.08787495503274936), (33, 0.09004503377814962), (34, 0.0), (35, 0.10283867552865913), (36, 0.17348855898381338), (37, 0.07798128673650545), (38, 0.0769657842

In [1]:
import pickle

In [31]:
pickle.dump(new_df.to_dict(), open('movie_list_dict.pkl', 'wb'))

In [32]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))